# Load data

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import clear_output
from datetime import datetime
import pickle
from datetime import datetime
import math
import scanpy as sc
from adjustText import adjust_text

pd.options.display.max_rows = 2000

today = datetime.today().strftime('%Y-%m-%d')

mpl.rcParams['svg.fonttype'] = 'none'  # 'none' = keep text as text objects


# Optional: improve SVG precision

mpl.rcParams['svg.hashsalt'] = ''  # consistent hashes for reproducibility

today = datetime.today().strftime('%Y-%m-%d')

mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=[ '#009736','#EE2A35',"#3f488aff",
                                                    "#f79a00ff", "#cf1100ff", "#81a5bfff",
                                                    "#f9bd00ff","#547200ff", "#bfd8cdff"]) 

In [ ]:
from module.misc import sample_name_import

name_dir = "circa-SD"
# name_dir = "all-samples"

samples, samples_ids = sample_name_import(name_dir)

print(len(samples))
print(samples)

In [ ]:
dir_notebook = '../notebook'
# dir_notebook = 'D:\Jupyter_notebook\Xenium_jupyter_notebook'


In [ ]:
adata = sc.read_h5ad(f'{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz')

In [ ]:
df = pd.read_parquet(f'{dir_notebook}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
## SD dendro order
order_list = ['STR D2 Gaba', 'STR D1 Gaba', 'DG Glut', 'CA1 ProS Glut', 'CA2 FC IG Glut', 'CA3 Glut',
               'LSX Gaba', 'CEA Gaba', 'STR PAL Gaba', 'COAa PAA MEA Glut', 'NLOT Glut', 'L2 3 IT PIR ENTl Glut',
                 'CLA EPd CTX Glut', 'L6 CT CTX Glut', 'L6b CTX Glut', 'L5 NP CTX Glut', 'L5 ET CTX Glut',
                   'L2 3 IT RSP Glut', 'L4 5 IT CTX Glut', 'L6 IT CTX Glut', 'Lamp5 Gaba', 'Vip Gaba', 'Sncg Gaba',
                     'Sst Gaba', 'STR Gaba', 'Pvalb Gaba', 'RT ZI Gaba', 'AD Glut', 'PT Glut', 'RE Glut', 'CM Glut',
                       'SMT Glut', 'VP Glut', 'LD Glut', 'VM MD Glut', 'AV Glut', 'MH Glut', 'TRS BAC Glut', 'BAC Glut',
                         'MPO Glut', 'PAL STR Gaba Chol', 'PVT Glut', 'MEA Glut', 'BST Glut', 'SPA Glut', 'AHN Glut',
                           'LHA Glut', 'LH Glut', 'PVH Glut', 'SCH Gaba', 'DG PIR Ex IMN', 'OB STR CTX IMN', 'Microglia',
                             'Endothelial', 'SMC','Pericyte', 'Choroid', 'ABC', 'VLMC', 'Tanycyte', 'Ependymal',
                               'Astro TE', 'OPC', 'Oligodendrocyte']

# Cell Population

In [ ]:
grouped = df.groupby(['run','sample','region_automap_name'])['cell_type_final'].value_counts(dropna=True)
grouped.to_csv(f'data/cell_count_region_{name_dir}.csv')

In [ ]:
import math
df = pd.read_csv(f'data/cell_count_region_{name_dir}.csv')

df = df[df['count'] > 10]

df_nbcells_av = []
df_nbcells_sem = []
df_nbcells_pct = []

all_regions = df['region_automap_name'].unique()
all_celltypes = df['cell_type_final'].unique()

for region in all_regions:
    dat_av = pd.DataFrame()
    dat_sem = pd.DataFrame()
    dat_pct = pd.DataFrame()
    
    for cell in all_celltypes:
        temp_av = df[(df['cell_type_final']== cell) & (df['region_automap_name'] == region)].groupby('run')['count'].mean()
        tempdf = pd.DataFrame(data = {cell : temp_av})
        dat_av = pd.concat([dat_av, tempdf], axis=1)

        temp_percent = df[(df['cell_type_final']== cell)& (df['region_automap_name'] == region) ].groupby('run')['count'].sum() / df[df['region_automap_name'] == region].groupby('run')['count'].sum() * 100
        tempdf = pd.DataFrame(data = {cell : temp_percent}) 
        dat_pct = pd.concat([dat_pct, tempdf], axis=1)  

        temp_sem = df[(df['cell_type_final']== cell) & (df['region_automap_name'] == region)].groupby('run')['count'].std() / df[(df['cell_type_final']== cell) & (df['region_automap_name'] == region)].groupby('run')['count'].count()
        tempdf = pd.DataFrame(data = {cell : temp_sem})
        dat_sem = pd.concat([dat_sem, tempdf], axis=1)
    
    df_nbcells_sem.append(dat_sem)
    df_nbcells_av.append(dat_av)
    df_nbcells_pct.append(dat_pct)

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/celltype_average.xlsx', engine='xlsxwriter')
for j in range(0,len(all_regions)):
    df_nbcells_av[j].to_excel(writer, sheet_name=all_regions[j], index=False)
writer.close()

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/celltype_std.xlsx', engine='xlsxwriter')
for j in range(0,len(all_regions)):
    df_nbcells_sem[j].to_excel(writer, sheet_name=all_regions[j], index=False)
writer.close()

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/celltype_pct.xlsx', engine='xlsxwriter')
for j in range(0,len(all_regions)):
    df_nbcells_pct[j].to_excel(writer, sheet_name=all_regions[j], index=False)
writer.close()

In [ ]:
df_average = pd.read_excel(f'{dir_notebook}/analysis/{name_dir}/celltype_average.xlsx', sheet_name=None)
df_std = pd.read_excel(f'{dir_notebook}/analysis/{name_dir}/celltype_std.xlsx', sheet_name=None)
df_pct = pd.read_excel(f'{dir_notebook}/analysis/{name_dir}/celltype_pct.xlsx', sheet_name=None)

df_average = {key: value for key, value in sorted(df_average.items())}
df_std = {key: value for key, value in sorted(df_std.items())}
df_pct = {key: value for key, value in sorted(df_pct.items())}


df_average.pop('VLMC')
df_average.pop('Ependymal')
# df_average.pop('CHOR')
# df_average.pop('Tanycyte')

# df_std.pop('Tanycyte')
df_std.pop('VLMC')
# df_std.pop('CHOR')
df_std.pop('Ependymal')

# df_pct.pop('Tanycyte')
df_pct.pop('VLMC')
# df_pct.pop('CHOR')
df_pct.pop('Ependymal')
# df_pct.pop('AD')

In [ ]:
df_average['SCH']

In [ ]:
df[(df['cell_type_final']== "Astro TE") & (df['region_automap_name'] == "SCH")].groupby('run')['count'].mean()

In [ ]:
group = ['SD1','circa4']

for key in df_pct.keys():
    if key == 'PT':
        continue
    elif key == 'Tanycyte':
        continue
    elif key == 'B':
        continue
    elif key == 'A':
        continue
    elif key == 'BST':
        continue
    elif key == 'AD':
        continue
    print(key)
    df_average[key].dropna(axis=1, inplace = True)
    df_std[key] = df_std[key][df_average[key].columns]
    df_average[key].index = group
    df_std[key].index = group

    plt.figure(figsize=(5,8))
    plt.barh(y=df_average[key].T.index,
         width=df_average[key].T[group[0]],
           height= 0.45, align='edge', label = group[0],edgecolor='black',
           xerr= df_std[key].T[group[0]] )
    
    plt.barh(y=df_average[key].T.index,
         width=df_average[key].T[group[1]],
           height= -0.45, align='edge', label = group[1],edgecolor='black',
           xerr= df_std[key].T[group[1]]
           )

    
    plt.xlabel('Average cell number')
    plt.legend()
    plt.title(f'Average cell population : {key}')
    plt.savefig(f'Gallery/{today}/celltype_region/{name_dir}_cellnb_{key}.svg')
    clear_output()

In [ ]:
len(df_pct[key])

In [ ]:
for key in ['SCH']:
    if len(df_pct[key]) == 1:
        continue
    df_pct[key] = df_pct[key][df_pct[key].select_dtypes(np.number).gt(0.5)]
    df_pct[key].dropna(axis=1, inplace = True)
    df_std[key] = df_std[key][df_pct[key].columns]
    df_pct[key].index = ['NS','SD']
    df_std[key].index = ['NS','SD']

    plt.figure(figsize=(5,8))
    plt.barh(y=df_pct[key].T.index,
         width=df_pct[key].T['NS'],
           height= 0.45, align='edge', label = 'NS',edgecolor='black',
           )
    plt.barh(y=df_pct[key].T.index,
         width=df_pct[key].T['SD'],
           height= -0.45, align='edge', label = 'SD',edgecolor='black',
           )
    plt.xlabel('Percentage cells')
    plt.legend()
    plt.title(f'Average cell population : {key}')
    # plt.savefig('Gallery/circa-SD_cellnb.svg')
    clear_output()

# Cell population dendro order

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
cell_nb = pd.read_csv(f'{dir_notebook}/analysis/{name_dir}/summary_cell_number.csv')
cell_nb.sample()

In [ ]:
from module.misc import cell_class

dict_temp = cell_class()

cell_nb['cell_class'] = cell_nb['cell_type_final'].apply(lambda x: dict_temp[x] if x in dict_temp.keys() else "Neuron")
cell_nb.sample()

cell_nb["Celltype_order"] = pd.Categorical(cell_nb['cell_type_final'], categories=order_list, ordered = True)
cell_nb.sort_values('Celltype_order', inplace= True)

In [ ]:
param = 'run'
group1 = "circa4"
group2 = 'SD1'

plt.figure(figsize=(2,10))
plt.barh(y=cell_nb['cell_type_final'][cell_nb[param]==group1],
         width=cell_nb[cell_nb[param]==group1]['count'],
           height = 0.5, align='edge', edgecolor ='black')
plt.barh(y=cell_nb['cell_type_final'][cell_nb[param]==group2],
         width=cell_nb[cell_nb[param]==group2]['count'],
           height= -0.5, align='edge', edgecolor ='black')
plt.yticks(ticks=cell_nb['cell_type_final'][cell_nb[param]==group2], rotation=0)
plt.xscale('log')
plt.savefig(f'Gallery/{today}/{name_dir}_cellnb.svg')

## Cell pop hierarchical order

In [ ]:
df = df[~(df['cell_type_final'] == 'Undefined')]
# df = df[~(df['cell_type_final'] == 'AD Glut')]

In [ ]:
cell_pop = df.groupby(['cell_type_final','run'])["ZT"].count()
cell_pop.head()

In [ ]:
# cell_pop = pd.read_csv(f'../notebook/analysis/{name_dir}/{name_dir}_cellpop.csv')
# circa_pct = pd.read_csv('../notebook/analysis/circa-SD/circa-SD_circa-percent.csv')

In [ ]:
cell_pop = pd.DataFrame(data = {"cell_type_final" : cell_pop.index,
                                "count" : cell_pop.values})
# cell_pop.rename(columns={"ZT" : "count"})


In [ ]:
from module.misc import cell_class

cell_pop[['celltype','genotype']] = cell_pop['cell_type_final'].apply(pd.Series)

class_dict = cell_class()
cell_pop['cell_class'] = 'Neuron'
cell_pop['cell_class'] = cell_pop['celltype'].map(class_dict)
cell_pop['cell_class'] = cell_pop['cell_class'].fillna('Neuron')

cell_pop

In [ ]:
# cell_pop['cell_type_final'] = cell_pop.index
cell_pop.sort_values(by='cell_type_final', inplace=True, ignore_index=True)
# circa_pct.sort_values(by='cell_type_final', inplace=True, ignore_index=True)


In [ ]:
from module.misc import genes_list

clock_genes = genes_list('panel_5k')
df_CG = df.filter(clock_genes, axis = 1)

In [ ]:
df_CG['cell_type_final'] = df['cell_type_final']
df_CG.sample() 

In [ ]:
grouped = df.groupby('cell_type_final')[clock_genes].mean()

In [ ]:
order_list = ['STR D2 Gaba', 'STR D1 Gaba', 'DG Glut', 'CA1 ProS Glut', 'CA2 FC IG Glut', 'CA3 Glut',
               'LSX Gaba', 'CEA Gaba', 'STR PAL Gaba', 'COAa PAA MEA Glut', 'NLOT Glut', 'L2 3 IT PIR ENTl Glut',
                 'CLA EPd CTX Glut', 'L6 CT CTX Glut', 'L6b CTX Glut', 'L5 NP CTX Glut', 'L5 ET CTX Glut',
                   'L2 3 IT RSP Glut', 'L4 5 IT CTX Glut', 'L6 IT CTX Glut', 'Lamp5 Gaba', 'Vip Gaba', 'Sncg Gaba',
                     'Sst Gaba', 'STR Gaba', 'Pvalb Gaba', 'RT ZI Gaba', 'AD Glut', 'PT Glut', 'RE Glut', 'CM Glut',
                       'SMT Glut', 'VP Glut', 'LD Glut', 'VM MD Glut', 'AV Glut', 'MH Glut', 'TRS BAC Glut', 'BAC Glut',
                         'MPO Glut', 'PAL STR Gaba Chol', 'PVT Glut', 'MEA Glut', 'BST Glut', 'SPA Glut', 'AHN Glut',
                           'LHA Glut', 'LH Glut', 'PVH Glut', 'SCH Gaba', 'DG PIR Ex IMN', 'OB STR CTX IMN', 'Microglia',
                             'Endothelial', 'SMC','Pericyte', 'Choroid', 'ABC', 'VLMC', 'Tanycyte', 'Ependymal',
                               'Astro TE', 'OPC', 'Oligodendrocyte']

# order_list = ['Oligodendrocyte', 'OPC', 'Astro NT', 'Astro TE', 'Ependymal', 'Pineal Glut', 'Tanycyte', 'CHOR', 'VLMC', 'Endothelial', 'Pericyte', 'Microglia', 'CA1 ProS Glut', 'CA2 FC IG Glut', 'CA3 Glut', 'DG Glut', 'L23 PIR ENTl Glut', 'MEA Glut', 'LA Glut', 'NLOT Glut', 'L23 CTX Glut', 'L23 RSP Glut', 'L4 CTX Glut', 'L6 CTX Glut', 'L5 CTX Glut', 'SUB ProS Glut', 'L6b CTX Glut', 'AD Glut', 'AV Glut', 'TH Glut', 'SN Dopa', 'LHA Glut', 'MB Glut', 'PAG Glut', 'HY Glut', 'LH Glut', 'VMH Glut', 'MM Glut', 'PVT Glut', 'PF Glut', 'APN Glut', 'SC Glut', 'MH Glut', 'BST Glut', 'LSX Gaba', 'SCH Gaba', 'Sst Gaba', 'MEA Gaba', 'BST Gaba', 'HY GABA', 'ARH GABA', 'Lamp5 Gaba', 'Vip Gaba', 'STR Gaba', 'STRv PAL Gaba', 'LGv Gaba', 'PRT Gaba', 'SC Gaba', 'ZI Gaba', 'SN Gaba', 'RT ZI GABA', 'Pvalb Gaba', 'STR D1D2 Gaba']



# cell_pop["Celltype_order"] = pd.Categorical(cell_pop["cell_type_final"], categories=order_list, ordered = True)
cell_pop["Celltype_order"] = pd.Categorical(cell_pop["celltype"], categories=order_list, ordered = True)

cell_pop.sort_values('Celltype_order', inplace= True)

# grouped_sort = grouped.reindex(order_list)
# grouped_sort

In [ ]:
cell_pop.head()

In [ ]:
### One bar for all
import seaborn as sns

plt.figure(figsize=(2,10))

plt.barh(y=cell_pop['cell_type_final'],
         width=cell_pop['count'],
           height = 0.8, align='edge', edgecolor ='black')
plt.xscale('log')
plt.grid(axis='x')
plt.ylim(-1,len(cell_pop))


# plt.savefig(f'Gallery/{today}/{name_dir}_cellpop.svg', dpi=300, format = "svg",transparent = True)

In [ ]:
### One bar epr genotype
import seaborn as sns



temp = cell_pop#[cell_pop['cell_class']=="Neuron"]
# temp.sort_values(by="count", inplace = True)

plt.figure(figsize=(2,10))

plt.barh(y=temp[temp['genotype']==group1]['celltype'],
         width=temp[temp['genotype']==group1]['count'],
           height = 0.4, align='edge', edgecolor ='black')
plt.barh(y=temp[temp['genotype']==group2]['celltype'],
         width=temp[temp['genotype']==group2]['count'],
           height = -0.4, align='edge', edgecolor ='black')
plt.xscale('log')
plt.grid(axis='x')
plt.ylim(-1,len(temp)/2)


plt.savefig(f'Gallery/{today}/{name_dir}_cellpop.svg', dpi=300, format = "svg",transparent = True)

# Expressed genes

In [ ]:
df_NS = df[(df['run']=='circa4')]
df_SD = df[(df['run']=='SD1')]
del df

In [ ]:
from module.misc import genes_list

clock_genes = genes_list('panel_5k')
dict_pos = {}

grouped = df_NS.groupby('cell_type_final')


for cell_type, group in grouped:
    # group.reindex()
    gene_presence_pct = group[clock_genes].mean()
    percent_cell_expressing = (group[clock_genes] != 0).sum(axis=0) / group[clock_genes].count()
    expressed_genes_pct = set(percent_cell_expressing[percent_cell_expressing >= 0.1].index.tolist())
    expressed_genes = set(gene_presence_pct[gene_presence_pct >= 0.01].index.tolist())
    total_expressed = expressed_genes.intersection(expressed_genes_pct)
    dict_pos[cell_type] = total_expressed

len(dict_pos['Vip Gaba'])

In [ ]:
len(gene_presence_pct[gene_presence_pct >= 0.01].index.tolist()),len(percent_cell_expressing[percent_cell_expressing >= 0.1].index.tolist())

In [ ]:
from module.misc import genes_list

clock_genes = genes_list('panel_5k')
dict_pos = {}

grouped = df_SD.groupby('cell_type_final')


for cell_type, group in grouped:
    # group.reindex()
    gene_presence_pct = group[clock_genes].mean()
    percent_cell_expressing = (group[clock_genes] != 0).sum(axis=0) / group[clock_genes].count()
    expressed_genes_pct = set(percent_cell_expressing[percent_cell_expressing >= 0.15].index.tolist())
    expressed_genes = set(gene_presence_pct[gene_presence_pct >= 0.01].index.tolist())
    total_expressed = expressed_genes.intersection(expressed_genes_pct)
    dict_pos[cell_type] = total_expressed

len(dict_pos['Vip Gaba'])

In [ ]:
print(list(dict_pos['SCH Gaba']))

In [ ]:
gene_nb = [f'{key} : {len(dict_pos[key])}' for key in dict_pos.keys()]
gene_nb

In [ ]:
gene_nb.to_csv(f'../notebook/analysis/{name_dir}/{name_dir}-genexpression-summary.csv')

In [ ]:
for key in dict_pos.keys():
    temp = set(dict_pos[key])

# Ratio of cell expressing genes

In [ ]:
# NS
from module.misc import genes_list
import pandas as pd
import pickle

with open(f"{dir_notebook}/analysis/circa-SD/cycling_genes_database.pickle", "rb") as handle:
    dict_all_cycling = pickle.load(handle)

clock = genes_list('clock')

dfdf = pd.DataFrame(index=clock)

for cell in dict_all_cycling.keys():
    
    dfdf[cell] = dict_all_cycling[cell]['DEG'][dict_all_cycling[cell]['DEG']['group']=="circa4"]['pct_nz_group'][clock].values

In [ ]:
import seaborn as sns

sns.clustermap(dfdf.T, cmap = 'Blues', vmin= 0,
                col_cluster=False,row_cluster=False, cbar = True, cbar_pos=None,figsize=(5,10),
                )
plt.title('Percentage of cells expressing CG')
plt.savefig(f'Gallery/{today}/NS_corr_cyclinggenes_ordernbcycgenes_otherrange.svg',dpi=300)

In [ ]:
# SF
from module.misc import genes_list
import pandas as pd
import pickle

with open(f"{dir_notebook}/analysis/circa-SD/cycling_genes_database.pickle", "rb") as handle:
    dict_all_cycling = pickle.load(handle)

clock = genes_list('clock')

dfdf = pd.DataFrame(index=clock)

for cell in dict_all_cycling.keys():
    
    dfdf[cell] = dict_all_cycling[cell]['DEG'][dict_all_cycling[cell]['DEG']['group']=="SD1"]['pct_nz_group'][clock].values

import seaborn as sns

sns.clustermap(dfdf.T, cmap = 'Blues', vmin= 0,
                col_cluster=False,row_cluster=False, cbar = True, cbar_pos=None,figsize=(5,10),
                )
plt.title('Percentage of cells expressing CG')
plt.savefig(f'Gallery/{today}/SF_corr_cyclinggenes_ordernbcycgenes_otherrange.svg',dpi=300)